In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

## Note 

- To run this notebook, run this first: `python -m src.data.restructure_entrainment_coding_data`
- It is recommended to run `python -m src.data.restructure_entrainment_coding_data --output_csv` for real analysis

In [2]:
import pandas as pd

dogs_df = pd.read_excel("baskets_dogs_data/dogs.xlsx")
baskets_df = pd.read_excel("baskets_dogs_data/baskets.xlsx")

In [6]:
dogs_df.head(1)

,Round,Order,TargetObjectIndex,Pair_20,Pair_21,Pair_22,Pair_23,Pair_24,Pair_34,Pair_35,Pair_37,Pair_43,Pair_45
0,1,Dog 1,2,"D: Mike, do you know anything about dogs?\n \n...","D: Okie dokie\nUm, number one is a dog\nIt's ...","D: Ok, number one--it's uh--it's a dog with t...","D: Ok (laughs)\nOk, this dog is standing upri...","D: Okay\nUm, dog number one is uh--(laughs)\n...",D: In the middle of the dog is black\nIt's fa...,D: Picture number one\n\nM: Mm-hmm\n\nD: Is...,D: Uh…the first dog's uh...t-t-t-t (clicks to...,"M: Ok, and what color is it?\n\nD: It has blac...","D: Ok, Joanne…the first dog…um it's standing ..."


In [7]:
dogs_df.columns

Index(['Round', 'Order', 'TargetObjectIndex', 'Pair_20', 'Pair_21', 'Pair_22',
       'Pair_23', 'Pair_24', 'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43',
       'Pair_45'],
      dtype='object')

In [16]:
def get_pair_transcript_and_entrainment_coding_data(
        object_category: str, 
        pair_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    if object_category == "dogs":
        df = dogs_df
    elif object_category == "baskets":
        df = baskets_df
    else:
        raise ValueError("object_category must be either 'dogs' or 'baskets'")

    assert pair_col in df.columns, f"{pair_col} not found in dataframe columns"

    pair_ix = pair_col.split("_")[-1]
    transcript_df = df[['Round', 'Order', 'TargetObjectIndex', pair_col]]
    entrainment_coding_df = pd.read_json("baskets_dogs_data/restructured_entrainment_coding_data/"
                                         f"{object_category}/pair{pair_ix}.json")
    
    return transcript_df, entrainment_coding_df

## Check one example

Hypothesis:

- Card1 to Card10 correspond to the card orders in the first 1
- The descriptions from `entrainment_coding_df` are extracted from `transcript_df` (extractive summarization)

In [32]:
from src.data.utils import pretty_print

pair_col = "Pair_23"
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="dogs", pair_col=pair_col
)
transcript_df.shape, entrainment_coding_df.shape

((40, 4), (4, 10))

In [19]:
pretty_print(entrainment_coding_df.T)

,Round1,Round2,Round3,Round4
Card1,"[standing upright, patch of black around stomach and back, area, legs are more a camel color, orangish color, yellowish, color towards tail, black tip on tail]","[black patch, first dog we looked at]","[first dog we ever looked at, black patch, on stomach]","[first dog we looked at, black patch]"
Card2,"[furry, turning towards his butt, white legs, patch of black on, his butt, looking at his tail, standing, black outline, yellow, on bottom]","[looks like he's biting his butt, yellow outline at, bottom, black on top]","[yellow bottom, black top]","[looks like he's biting his butt, yellow on, bottom, black on top]"
Card3,"[sitting down, brown collar, surrounded by green, looking, back at his tail]","[looking at his tail, sitting down, green outline]","[sitting down, looking towards, green, outline]","[green outline, looks at his tail]"
Card4,"[really short, weiner]","[short, weiner]",[weiner],[weiner]
Card5,"[poodle, all black, shiny collar thing, circular, silver color]","[poodle, black, circular thing on collar]",[circle thing on collar],[poodle with circle thing on collar]
Card6,"[Doberman, shiny black, outlined in green]","[Doberman, shiny all over, black]","[Doberman, shiny all over, black]","[shiny black, Doberman]"
Card7,"[cocker spaniel, longish hair, looking up, grass around,, ears are droopy, not high, brown fur, greenish background]","[brown hair, ears droop, cocker spaniel]","[cocker spaniel, tan color, droopy ears, curly, hair, wavy, looks like human hair, small]","[cocker spaniel, looks like he has, human hair]"
Card8,"[bulldog, legs look like he's digging into ground, puppy,, sad face]","[sad puppy face, legs are digging into ground, behind him, bulldog]",[bulldog],[bulldog]
Card9,"[wolf dog, white tail]",[wolf],[wolf],[wolf]
Card10,"[short, beard, tail pointing up, all black]","[""the one that gave us trouble before"", beard,, curly hair, black, ears pointing upwards, ""he, was number ten""]","[""the one with the ears"", tail sticking up,, black, short]","[little, furry, black, ears point up, tail, points up]"


In [22]:
TargetObjectIndices = transcript_df[transcript_df['Round'] == 1].TargetObjectIndex
TargetObjectIndices

0     2
1     8
2     4
3    10
4     6
5     3
6     7
7     1
8     9
9     5
Name: TargetObjectIndex, dtype: int64

In [ ]:
pretty_print(transcript_df[transcript_df.TargetObjectIndex == TargetObjectIndices[0]])

,Round,Order,TargetObjectIndex,Pair_23
0,1,Dog 1,2,"D: Ok (laughs)Ok, this dog is standing upright(laughs)M: OkD: Um, he's got a patch of black around his stomach and back area, and his legs are more like a camel colorM: Um, like a orangish color?D: YeahM: Um, like towards his tail, is it like a y--uh--yellowish…color?D: Yeah, but he has like a black tip on his tailM: Yup got it, ok"
11,2,Dog 2,2,"D: Number two--he's got the black patchHe's the first dog we looked atM: Oh, ok"
28,3,Dog 9,2,"D: *Top*YeahNumber nine is the first dog we ever looked at, with the black patch on his stomachHe's got--M: Right, ok, I got him"
39,4,Dog 10,2,"D: And number ten is…the first dog we looked atWith the black patch, and...M: Ok*Got it*D: *Ok*"


In [ ]:
from tqdm import tqdm
from src.llms.openai import get_response

prompt_tmp = """
You will be given a text description and a excerpt from a conversation transcript. \
Your task is to verify if the description is extracted from the transcript. \
Compare the description and the transcript excerpt carefully. And output "YES" if \
all points in the description are found in the transcript excerpt, otherwise output "NO". \
Do not provide any additional explanation.

### Description

{description}

### Transcript 

{transcript}

### Answer
""".strip()

out = []
progress_bar = tqdm(total=2 * 10 * 10 * 4, position=0, 
                    leave=True, desc="Processing Pairs")

for category in ["dogs", "baskets"]:

    for pair_col in dogs_df.columns.to_list()[3:]:
        transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
            object_category=category, pair_col=pair_col
        )

        TargetObjectIndices = transcript_df[transcript_df['Round'] == 1].TargetObjectIndex

        for ix, target_obj_index in enumerate(TargetObjectIndices):
            transcripts = transcript_df[transcript_df.TargetObjectIndex 
                                        == target_obj_index][pair_col]
            descriptions = entrainment_coding_df[f"Card{ix+1}"]

            assert len(transcripts) == len(descriptions), \
                "Mismatch in number of rounds between transcript and entrainment coding data"

            for j, (transcript, description) in enumerate(zip(transcripts, descriptions)):
                description = " ".join(description)
                prompt = prompt_tmp.format(
                    description=description,
                    transcript=transcript
                )
                _, response = get_response(
                    prompt,
                    model_name="gpt-4o-mini",
                )
                out.append(
                    {
                        "Object Category": category,
                        "Participant Pair": pair_col,
                        "Round": j + 1,
                        "TargetObjectIndex": target_obj_index,
                        "Model Response": response,
                    }
                )
                progress_bar.update(1)

Processing Pairs:   1%|▏         | 1/80 [00:16<21:11, 16.09s/it]


Processing Pairs: 800it [29:55,  2.21s/it]                       

In [50]:
out_df = pd.DataFrame(out)
out_df["Model Response"].value_counts()

Model Response
YES    790
NO      10
Name: count, dtype: int64

In [59]:
out_df["Model Response"].to_csv("notebooks/outputs/entrainment_coding_verification_responses.csv", index=False)

## Manual Inspection


In [60]:
out_df[out_df["Model Response"] == "NO"]

,Object Category,Participant Pair,Round,TargetObjectIndex,Model Response
16,dogs,Pair_20,1,6,NO
33,dogs,Pair_20,2,9,NO
65,dogs,Pair_21,2,7,NO
73,dogs,Pair_21,2,9,NO
144,dogs,Pair_23,1,7,NO
414,baskets,Pair_20,3,5,NO
417,baskets,Pair_20,2,10,NO
440,baskets,Pair_21,1,9,NO
530,baskets,Pair_23,3,7,NO
752,baskets,Pair_43,1,2,NO


### Dogs

In [61]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="dogs", pair_col="Pair_20"
)
pretty_print(transcript_df[transcript_df.TargetObjectIndex == 6])

,Round,Order,TargetObjectIndex,Pair_20
4,1,Dog 5,6,D: Alright Good The next one is the poodle M: Alright (laughter)
15,2,Dog 6,6,"D: Uh, the next is the--looks like the black poodle? M: Alright"
20,3,Dog 1,6,D: First is the poodle M: Alright
35,4,Dog 6,6,D: The poodle? M: Alright


In [ ]:
# Round 1 needs double check, but round 2 looks correct.
pretty_print(entrainment_coding_df[["Card5"]].loc[["Round1", "Round2"]])

,Card5
Round1,"[looking towards left, poodle]"
Round2,[black poodle]


In [68]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="dogs", pair_col="Pair_21"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex.isin([7, 9])) & 
                           (transcript_df.Round.isin([1, 2]))])


,Round,Order,TargetObjectIndex,Pair_21
6,1,Dog 7,7,"D: Oh okNumber seven looks like a…golden retriever but in a poodle version?It's like, uh...it's--it's--it's stretching its neck out, upAnd it's very furry, and it's brownM: AlrightD: It's very smallM: Does it have, uh, some…black spots?D: Uh, no, no black spotsM: Uh…What are the ears doing?D: Ears are down, and they're very longM: OkD: And it's very furry, and it's facing to the left, with its h--face sticking upM: His face sticking up--oh okD: You got it?M: Yup"
8,1,Dog 9,9,D: *Hmm--ok*Number nine is a...it looks like a Siberian husky**It** looks like a ***wolf***M: **Ok*****Yup***I got it
16,2,Dog 7,7,D: The seventh one is the…golden retriever with the face…smelling?You remember?M: Yeah
18,2,Dog 9,9,"D: And his head is downUh, the ninth one is a Siberian huskyM: *Yup*"


In [ ]:
# Card7, Round 2: double check
# Card9, Round 2: double check
# possible: the segmentation of the transcript into 10 parts has some issues
pretty_print(entrainment_coding_df[["Card7", "Card9"]].loc[["Round1", "Round2"]])

,Card7,Card9
Round1,"[looks like golden retriever but in poodle version, stretching, its neck out, up, very furry, brown, very small, ears are down,, very long, facing to left, face sticking up, smelling something,, no tail, can't see tail]","[Siberian husky, wolf]"
Round2,"[golden retriever with face smelling, no, tongue, ears are down, nose is up, body, extending towards head, stretching its, neck out]","[Siberian husky, looks like a wolf]"


In [ ]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="dogs", pair_col="Pair_23"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex == 7) &
                           (transcript_df.Round.isin([1]))])

,Round,Order,TargetObjectIndex,Pair_23
6,1,Dog 7,7,"D: Ok, number seven is--he looks like a cocker spanielHe's got like longish hair, and he's looking up as *if--*M: *Ok,* and the gru--and the grass around himD: YeahM: Ok"


In [ ]:
# need to double check
pretty_print(entrainment_coding_df[["Card7"]].loc[["Round1"]])

,Card7
Round1,"[cocker spaniel, longish hair, looking up, grass around,, ears are droopy, not high, brown fur, greenish background]"


### Baskets

In [75]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="baskets", pair_col="Pair_20"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex.isin([5])) &
                           (transcript_df.Round.isin([1,3]))])

,Round,Order,TargetObjectIndex,Pair_20
3,1,Basket 4,5,"D: If you go along the whole thing The next one is a very tight weave, without a handle It's a very large bowl Uh... Let's see Nothing else... Uh, the bottom looks like a solid piece of, I guess, bamboo or whatever they used to h--to mix it together M: Mm-hmm D: And it's, uh--it's very large, compared to...uh, it's just a large bowl it looks like But, uh, it's got a tight weave, and it's all--it's all uniform tan It's s--some, some spaces where that look like a darker color, a brown M: Alright D: But, uh, *that's about it* on that one M: *Alright I got that one* The next one?"
25,3,Basket 6,5,D: Number six is the very large bowl M: Ok


In [ ]:
# need to double check 
pretty_print(entrainment_coding_df[["Card4"]].loc[["Round1", "Round3"]])

,Card4
Round1,"[very tight weave, without handle, very large, bowl, bottom looks like solid piece of bamboo,, uniform tan, some spaces that look darker color,, brown]"
Round3,"[very large bowl, tight weave]"


In [77]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="baskets", pair_col="Pair_20"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex.isin([10])) &
                           (transcript_df.Round.isin([1,2]))])

,Round,Order,TargetObjectIndex,Pair_20
4,1,Basket 5,10,"D: The next one is a small basket, with a wide weave, and it's a got a red…weave through the whole thing *Uh...* M: *Like* a red ribbon? D: Yes M: Yeah I got that one D: Ok"
15,2,Basket 6,10,"D: Uh, the next one is the red ribbon... M: Alright, *next one*"


In [ ]:
# need to double check (striped)
pretty_print(entrainment_coding_df[["Card5"]].loc[["Round1", "Round2"]])

,Card5
Round1,"[small, wide weave, red weave through whole thing,, red ribbon]"
Round2,"[red ribbon, striped]"


In [79]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="baskets", pair_col="Pair_21"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex.isin([9])) &
                           (transcript_df.Round.isin([1]))])

,Round,Order,TargetObjectIndex,Pair_21
0,1,Basket 1,9,"D: Ok, uh… You ready? M: Yup D: (laughs) Ok, the first one is a long basket It looks like a rectangle It's probably the longest one in the picture It's square in shape Rectangular in shape M: Ok D: You got it? M: Yup"


In [ ]:
# need to double check (one handle in middle)
pretty_print(entrainment_coding_df[["Card1"]].loc[["Round1"]])

,Card1
Round1,"[long, rectangle, longest one in the picture, square, in shape, one handle in middle]"


In [81]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="baskets", pair_col="Pair_23"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex.isin([7])) &
                           (transcript_df.Round.isin([1,3]))])

,Round,Order,TargetObjectIndex,Pair_23
2,1,Basket 3,7,"D: And, um, number three--it's got a bigger weave than all the other baskets M: Ok *Like--* D: *It* looks sort of like a circle M: Ok and it's like--it's holes in it? D: Yeah M: Visible holes? D: Yeah, visible holes M: Ok"
26,3,Basket 7,7,"D: Number seven is the one with the gaps in it It looks like a circle, with a handle M: Oh, ok It has--it has a handle, right? Ok D: Yeah M: Alright"


In [ ]:
# need to double check (no gaps --> with the gaps? really big weave, biggest holes of all, the weaves?)
pretty_print(entrainment_coding_df[["Card3"]].loc[["Round1", "Round3"]])

,Card3
Round1,"[bigger weave than all the other baskets, circle, visible holes]"
Round3,"[with gaps, circle, no top,, really big weave, biggest holes of all, the weaves, circular handle]"


In [83]:
transcript_df, entrainment_coding_df = get_pair_transcript_and_entrainment_coding_data(
    object_category="baskets", pair_col="Pair_43"
)
pretty_print(transcript_df[(transcript_df.TargetObjectIndex.isin([2])) &
                           (transcript_df.Round.isin([1]))])

,Round,Order,TargetObjectIndex,Pair_43
8,1,Basket 9,2,"D: Number nine That's just s--oval shaped, kinda like a circle, it has a cover, and it has a handle that's kinda like--like half a triangle I mean not a triangle, a rectangle M: Is it um uh short? D: It's short M: Ok Got it"


In [ ]:
# looks correct
pretty_print(entrainment_coding_df[["Card9"]].loc[["Round1"]])

,Card9
Round1,"[oval shaped, circle, cover, handle that's half, a rectangle, short]"
